In [1]:
from stwhmm_jax_optimized import STWHMM
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path

sns.set_context("talk")

%matplotlib widget

In [2]:
observations = np.load("maze_clipped_spike_counts_pyr_only_Bilat_R02_20251106.npz")['observations']
print(observations.shape)

(130600, 201)


In [ ]:
M = 101
S = 10


model = STWHMM(
    M = M,
    S = S,
    s = None,
    sequence_lengths = None,
    T_mask = None,
    H_mask = None,
    random_state = 42,
    min_rate = 1e-6
)



results_dir = Path("fit_results")
results_dir.mkdir(exist_ok=True)

random_restarts = 10
best_result = None
best_log_likelihood = -np.inf
all_results = []

for i in range(random_restarts):

    print(f"\nRandom restart {i+1}/{random_restarts}")

    result = model.fit(
        observations,
        update_pi=True,
        n_iter=20,
        verbose=True,
        check = False
    )

    all_results.append(result)

    # Save this complete restart
    with open(results_dir / f"restart_{i:02d}.pkl", "wb") as f:
        pickle.dump(result, f)

    if result.log_likelihoods[-1] > best_log_likelihood:
        best_log_likelihood = result.log_likelihoods[-1]
        best_result = result



# Restore the parameters from the BEST restart
model.restore(best_result)

result = best_result

state_sequence = model.viterbi(observations)

Z_hat = state_sequence[0]
tau_hat = state_sequence[1]

T_hat = model.T_
H_hat = model.H_
Lambda_hat = model.Lambda_




Random restart 1/10
EM   1: loglik=-1159424.378702  delta=nan
EM   2: loglik=-1095348.059009  delta=64076.3
EM   3: loglik=-1004075.775484  delta=91272.3
EM   4: loglik=-969304.837789  delta=34770.9
EM   5: loglik=-955390.434143  delta=13914.4
EM   6: loglik=-947562.076537  delta=7828.36
EM   7: loglik=-942815.810375  delta=4746.27
EM   8: loglik=-939655.719971  delta=3160.09


RuntimeError: Zeta normalization failed at EM iteration 8; max |logsumexp|=1.001423727054e-06.